In [1]:
# ============================================================
# KAGGLE ONE-CELL TRAINING
# BURGERS-TYPE 8-FAMILY ICL
# DIRECT-LINEAR DATASET
# MIXED K PER OPTIMIZER BATCH -- PAD ALL TO 17
# ============================================================
#
# Controlled rerun of the established standard Burgers experiment.
# The dataset already stores model-facing coarse states after direct
# linear interpolation 50 -> 100. This script does NOT interpolate again.
#
# Batching experiment:
#   Each example independently samples K in {1,...,5}.
#   All B examples remain in ONE logical optimizer batch.
#   Every sequence is padded on the RIGHT to MAX_CTX_ROWS=17.
#   A src_key_padding_mask prevents valid tokens from attending to padding.
#   Therefore each optimizer step uses ONE forward/backward pass.
#
# Dataset:
# /kaggle/input/notebooks/xinyuanli913/dataset-box-ood/
#   all8families_box_linear_id_ampood_recorded.mat
#
# Output:
# /kaggle/working/burgers_box_linear_mixedK_pad17/
#
# This is TRAINING ONLY. No held-out test/OOD evaluation is performed.
# ============================================================

import time
import json
import random
from pathlib import Path

import numpy as np
import scipy.io as sio

import torch
import torch.nn as nn


# ============================================================
# 0. CONFIG
# ============================================================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAT_PATH = Path(
    "/kaggle/input/notebooks/xinyuanli913/"
    "dataset-box-ood/"
    "all8families_box_linear_id_ampood_recorded.mat"
)

RUN_TAG = "burgers_box_linear_mixedK_pad17_train1to5_testfixed5"
SAVE_DIR = Path("/kaggle/working/burgers_box_linear_mixedK_pad17")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MIN_DEMOS = 1
MAX_DEMOS = 5
TEST_FIXED_DEMOS = 5

NX = 100
N_TRAIN_VAR_PER_FAM = 70
N_TEST_VAR_PER_FAM = 30

BATCH_SIZE = 1000
EPOCHS = 300
STEPS_PER_EPOCH = 1200
PRINT_EVERY = 5

WEIGHT_DECAY = 1e-4
BASE_LR = 1e-4
MAX_LR = 2e-4
MIN_LR = 1e-5
WARMUP_FRAC = 0.08
GRAD_CLIP = 1.0

N_TRAIN_MONITOR_CASES = 512
EVAL_BATCH_SIZE = 256

D_MODEL = 96
N_HEAD = 4
N_LAYERS = 3
D_FF = 192
DROPOUT = 0.10

EPS = 1e-8

TYPE_X_DEMO = 0
TYPE_UC_DEMO = 1
TYPE_UF_DEMO = 2
TYPE_X_QUERY = 3
TYPE_UC_QUERY = 4
N_ROW_TYPES = 5
MAX_CTX_ROWS = 3 * MAX_DEMOS + 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("=" * 110)
print("BURGERS STANDARD DIRECT-LINEAR TRAINING -- MIXED K, PAD ALL TO 17")
print("=" * 110)
print("DEVICE   :", DEVICE)
print("MAT_PATH :", MAT_PATH)
print("SAVE_DIR :", SAVE_DIR)
print("K rule   : each episode independently samples K in {1,...,5}")
print("Batching  : pad every sequence to 17; ONE forward/backward per optimizer step")
print("=" * 110)

if not MAT_PATH.exists():
    raise FileNotFoundError(f"Dataset not found:\n{MAT_PATH}")


# ============================================================
# 1. LOAD DATASET
# ============================================================
mat = sio.loadmat(MAT_PATH)
required = [
    "operators", "x", "pair_id", "ic_base_params",
    "u0_id", "uc_id", "uf_id", "ic_params_id",
]
for key in required:
    if key not in mat:
        raise KeyError(f"Missing required dataset key: {key}")

operators = np.asarray(mat["operators"], dtype=np.float32)
x_grid = np.asarray(mat["x"], dtype=np.float32).squeeze()
pair_id = np.asarray(mat["pair_id"], dtype=np.int64)
ic_base_params = np.asarray(mat["ic_base_params"], dtype=np.float32)
u0_id = np.asarray(mat["u0_id"], dtype=np.float32)
uc_id = np.asarray(mat["uc_id"], dtype=np.float32)
uf_id = np.asarray(mat["uf_id"], dtype=np.float32)
params_id = np.asarray(mat["ic_params_id"], dtype=np.float32)

assert uc_id.shape == uf_id.shape == u0_id.shape
assert uc_id.ndim == 4
N_FAM, N_VAR, N_IC, NX_LOADED = uc_id.shape
assert N_FAM == 8
assert NX_LOADED == NX
assert operators.shape == (N_FAM, N_VAR, 3)
assert x_grid.shape == (NX,)
assert pair_id.shape == (N_FAM, N_VAR, N_IC)
assert ic_base_params.shape == (N_FAM, N_VAR, N_IC, 4)
assert params_id.shape == (N_FAM, N_VAR, N_IC, 4)

print("\nLoaded dataset:")
print("  uc / uf / u0 :", uc_id.shape)
print("  operators     :", operators.shape)
print("  x             :", x_grid.shape)
print("  families      :", N_FAM)
print("  operators/fam :", N_VAR)
print("  ICs/operator  :", N_IC)
print("  model row Nr  :", NX)
print("  preprocessing : dataset already direct-linear 50 -> 100")
print("                  NO interpolation in this training script")


# ============================================================
# 2. OPERATOR SPLIT -- SAME 70/30 RULE
# ============================================================
rng_ops = np.random.default_rng(SEED)
train_ops, test_ops = [], []
for fam in range(N_FAM):
    perm = rng_ops.permutation(N_VAR)
    train_var = perm[:N_TRAIN_VAR_PER_FAM]
    test_var = perm[
        N_TRAIN_VAR_PER_FAM:
        N_TRAIN_VAR_PER_FAM + N_TEST_VAR_PER_FAM
    ]
    train_ops.extend([fam * N_VAR + int(v) for v in train_var])
    test_ops.extend([fam * N_VAR + int(v) for v in test_var])

train_ops = np.asarray(train_ops, dtype=np.int64)
test_ops = np.asarray(test_ops, dtype=np.int64)
assert len(train_ops) == 560
assert len(test_ops) == 240
assert len(np.intersect1d(train_ops, test_ops)) == 0

print("\nOperator split:")
print("  train operators:", len(train_ops))
print("  test operators :", len(test_ops))
print("  per family     : 70 train / 30 held-out")


# ============================================================
# 3. FLATTEN FAMILY + VARIATION
# ============================================================
uc_flat = uc_id.reshape(N_FAM * N_VAR, N_IC, NX)
uf_flat = uf_id.reshape(N_FAM * N_VAR, N_IC, NX)
u0_flat = u0_id.reshape(N_FAM * N_VAR, N_IC, NX)
pair_id_flat = pair_id.reshape(N_FAM * N_VAR, N_IC)
operators_flat = operators.reshape(N_FAM * N_VAR, 3)


# ============================================================
# 4. NORMALIZATION -- TRAIN OPERATORS ONLY
# ============================================================
uc_train = uc_flat[train_ops]
uf_train = uf_flat[train_ops]

uc_mean = float(uc_train.mean())
uc_std = float(uc_train.std() + 1e-6)
uf_mean = float(uf_train.mean())
uf_std = float(uf_train.std() + 1e-6)
x_mean = float(x_grid.mean())
x_std = float(x_grid.std() + 1e-6)

x_n = ((x_grid - x_mean) / x_std).astype(np.float32)
uc_n = ((uc_flat - uc_mean) / uc_std).astype(np.float32)
uf_n = ((uf_flat - uf_mean) / uf_std).astype(np.float32)

print("\nNormalization from ID training operators only:")
print(f"  x  mean/std : {x_mean:.8f} / {x_std:.8f}")
print(f"  uc mean/std : {uc_mean:.8f} / {uc_std:.8f}")
print(f"  uf mean/std : {uf_mean:.8f} / {uf_std:.8f}")


# ============================================================
# 5. FIXED K=5 TRAIN-SIDE MONITOR CASES
# ============================================================
def make_fixed_cases(op_pool, n_cases, seed, fixed_demos=5):
    r = np.random.default_rng(seed)
    cases = []
    for _ in range(n_cases):
        op = int(r.choice(op_pool))
        idxs = r.choice(N_IC, size=fixed_demos + 1, replace=False).astype(np.int64)
        cases.append((op, idxs[:fixed_demos], int(idxs[-1])))
    return cases

monitor_cases = make_fixed_cases(
    train_ops,
    N_TRAIN_MONITOR_CASES,
    seed=SEED + 2,
    fixed_demos=TEST_FIXED_DEMOS,
)


# ============================================================
# 6. EFFICIENT DISTINCT IC SAMPLING
# ============================================================
def sample_distinct_ic_rows(rng, batch_size, n_pick, n_ic):
    out = rng.integers(0, n_ic, size=(batch_size, n_pick), dtype=np.int64)
    while True:
        s = np.sort(out, axis=1)
        bad = np.any(np.diff(s, axis=1) == 0, axis=1)
        n_bad = int(bad.sum())
        if n_bad == 0:
            break
        out[bad] = rng.integers(0, n_ic, size=(n_bad, n_pick), dtype=np.int64)
    return out


# ============================================================
# 7. MIXED-K LOGICAL BATCH SAMPLING
# ============================================================
rng_train = np.random.default_rng(SEED + 1000)


def sample_logical_batch_metadata(batch_size):
    """
    Sample one logical optimizer batch of B independent episodes.

    Each episode samples its own K ~ Uniform{1,...,5}.  We then sample
    its operator and K+1 distinct ICs (K demos + 1 query).

    Both the pad-to-17 and 5-group-exact implementations use this same
    metadata sampler so the *logical* batching rule is identical.
    """
    K_arr = rng_train.integers(
        MIN_DEMOS,
        MAX_DEMOS + 1,
        size=batch_size,
        dtype=np.int64,
    )
    ops = rng_train.choice(
        train_ops,
        size=batch_size,
        replace=True,
    ).astype(np.int64)

    # Store only the demo slots that are actually used by each episode.
    demo_idx = np.full(
        (batch_size, MAX_DEMOS),
        -1,
        dtype=np.int64,
    )
    q_idx = np.empty(batch_size, dtype=np.int64)

    # Sampling by K is only for efficient metadata construction.
    # It does NOT create separate optimizer steps.
    for K in range(MIN_DEMOS, MAX_DEMOS + 1):
        sel = np.flatnonzero(K_arr == K)
        if sel.size == 0:
            continue

        idxs = sample_distinct_ic_rows(
            rng_train,
            batch_size=sel.size,
            n_pick=K + 1,
            n_ic=N_IC,
        )
        demo_idx[sel, :K] = idxs[:, :K]
        q_idx[sel] = idxs[:, K]

    k_counts = {
        K: int(np.sum(K_arr == K))
        for K in range(MIN_DEMOS, MAX_DEMOS + 1)
    }

    return K_arr, ops, demo_idx, q_idx, k_counts


# ============================================================
# 7B. PAD-TO-17 TRAIN BATCH BUILDER
# ============================================================
def make_train_batch(batch_size):
    K_arr, ops, demo_idx, q_idx, k_counts = sample_logical_batch_metadata(
        batch_size
    )

    # Right-pad every episode to the maximum context length (17 rows).
    # True in src_key_padding_mask means "ignore this key/value position".
    mat_ctx = np.zeros(
        (batch_size, MAX_CTX_ROWS, NX),
        dtype=np.float32,
    )
    row_types = np.zeros(
        (batch_size, MAX_CTX_ROWS),
        dtype=np.int64,
    )
    padding_mask = np.ones(
        (batch_size, MAX_CTX_ROWS),
        dtype=bool,
    )

    lengths = (3 * K_arr + 2).astype(np.int64)
    target_n = np.empty((batch_size, NX), dtype=np.float32)

    for K in range(MIN_DEMOS, MAX_DEMOS + 1):
        sel = np.flatnonzero(K_arr == K)
        if sel.size == 0:
            continue

        ops_k = ops[sel]
        demos_k = demo_idx[sel, :K]
        q_k = q_idx[sel]
        S = 3 * K + 2

        row = 0
        for j in range(K):
            mat_ctx[sel, row, :] = x_n[None, :]
            row_types[sel, row] = TYPE_X_DEMO
            row += 1

            mat_ctx[sel, row, :] = uc_n[ops_k, demos_k[:, j], :]
            row_types[sel, row] = TYPE_UC_DEMO
            row += 1

            mat_ctx[sel, row, :] = uf_n[ops_k, demos_k[:, j], :]
            row_types[sel, row] = TYPE_UF_DEMO
            row += 1

        mat_ctx[sel, row, :] = x_n[None, :]
        row_types[sel, row] = TYPE_X_QUERY
        row += 1

        mat_ctx[sel, row, :] = uc_n[ops_k, q_k, :]
        row_types[sel, row] = TYPE_UC_QUERY
        row += 1
        assert row == S

        # Valid rows are [0, S); padding is [S, 17).
        padding_mask[sel, :S] = False
        target_n[sel, :] = uf_n[ops_k, q_k, :]

    # Sanity: the query-uc row (length-1) must always be valid.
    assert np.all(
        ~padding_mask[np.arange(batch_size), lengths - 1]
    )

    return {
        "mat_ctx": torch.from_numpy(mat_ctx).to(DEVICE, non_blocking=True),
        "row_types": torch.from_numpy(row_types).to(DEVICE, non_blocking=True),
        "padding_mask": torch.from_numpy(padding_mask).to(
            DEVICE, non_blocking=True
        ),
        "lengths": torch.from_numpy(lengths).to(DEVICE, non_blocking=True),
        "target_n": torch.from_numpy(target_n).to(DEVICE, non_blocking=True),
        "K_counts": k_counts,
    }


# ============================================================
# 8. FIXED MONITOR BATCH BUILDER
# ============================================================
def make_eval_batch(cases_subset):
    K = len(cases_subset[0][1])
    B = len(cases_subset)
    S = 3 * K + 2

    ops = np.asarray([int(c[0]) for c in cases_subset], dtype=np.int64)
    demo_idx = np.stack(
        [np.asarray(c[1], dtype=np.int64) for c in cases_subset], axis=0
    )
    q_idx = np.asarray([int(c[2]) for c in cases_subset], dtype=np.int64)

    mat_ctx = np.empty((B, S, NX), dtype=np.float32)
    row_types = np.empty((B, S), dtype=np.int64)

    row = 0
    for j in range(K):
        mat_ctx[:, row, :] = x_n[None, :]
        row_types[:, row] = TYPE_X_DEMO
        row += 1

        mat_ctx[:, row, :] = uc_n[ops, demo_idx[:, j], :]
        row_types[:, row] = TYPE_UC_DEMO
        row += 1

        mat_ctx[:, row, :] = uf_n[ops, demo_idx[:, j], :]
        row_types[:, row] = TYPE_UF_DEMO
        row += 1

    mat_ctx[:, row, :] = x_n[None, :]
    row_types[:, row] = TYPE_X_QUERY
    row += 1

    mat_ctx[:, row, :] = uc_n[ops, q_idx, :]
    row_types[:, row] = TYPE_UC_QUERY
    row += 1
    assert row == S

    target = uf_flat[ops, q_idx, :].astype(np.float32)

    return {
        "mat_ctx": torch.from_numpy(mat_ctx).to(DEVICE, non_blocking=True),
        "row_types": torch.from_numpy(row_types).to(DEVICE, non_blocking=True),
        "target": torch.from_numpy(target).to(DEVICE, non_blocking=True),
    }


# ============================================================
# 9. MODEL -- IDENTICAL STANDARD ARCHITECTURE
# ============================================================
class RowSequenceWholeBlockShiftTransformer(nn.Module):
    def __init__(
        self,
        nx=100,
        max_ctx_rows=17,
        n_row_types=5,
        d_model=96,
        nhead=4,
        num_layers=3,
        dim_ff=192,
        dropout=0.1,
    ):
        super().__init__()
        self.row_in = nn.Linear(nx, d_model)
        self.row_type_emb = nn.Embedding(n_row_types, d_model)
        self.pos_emb = nn.Embedding(max_ctx_rows, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            activation=torch.tanh,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.row_out = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.Tanh(),
            nn.Linear(d_model, nx),
        )

        self.register_buffer(
            "pos_ids",
            torch.arange(max_ctx_rows, dtype=torch.long),
            persistent=False,
        )

    def forward(self, mat_ctx, row_type_arr, src_key_padding_mask=None):
        _, S, _ = mat_ctx.shape
        h = self.row_in(mat_ctx)
        h = h + self.row_type_emb(row_type_arr)
        h = h + self.pos_emb(self.pos_ids[:S])[None, :, :]
        h = self.encoder(
            h,
            src_key_padding_mask=src_key_padding_mask,
        )
        return self.row_out(h)


model = RowSequenceWholeBlockShiftTransformer(
    nx=NX,
    max_ctx_rows=MAX_CTX_ROWS,
    n_row_types=N_ROW_TYPES,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=D_FF,
    dropout=DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("\nModel:")
print("  row width       :", NX)
print("  d_model         :", D_MODEL)
print("  heads           :", N_HEAD)
print("  layers          :", N_LAYERS)
print("  FF              :", D_FF)
print("  dropout         :", DROPOUT)
print("  trainable params:", f"{n_params:,}")
assert n_params == 255364, f"Unexpected parameter count: {n_params:,}"


# ============================================================
# 10. OPTIMIZER / SCHEDULER -- IDENTICAL
# ============================================================
TOTAL_STEPS = EPOCHS * STEPS_PER_EPOCH
WARMUP_STEPS = max(1, int(WARMUP_FRAC * TOTAL_STEPS))

opt = torch.optim.AdamW(
    model.parameters(),
    lr=MAX_LR,
    weight_decay=WEIGHT_DECAY,
)

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    opt,
    start_factor=BASE_LR / MAX_LR,
    end_factor=1.0,
    total_iters=WARMUP_STEPS,
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt,
    T_max=max(1, TOTAL_STEPS - WARMUP_STEPS),
    eta_min=MIN_LR,
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    opt,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[WARMUP_STEPS],
)

mse = nn.MSELoss()

print("\nOptimizer:")
print("  AdamW")
print("  batch size     :", BATCH_SIZE)
print("  epochs         :", EPOCHS)
print("  steps/epoch    :", STEPS_PER_EPOCH)
print("  total steps    :", TOTAL_STEPS)
print("  warmup steps   :", WARMUP_STEPS)
print("  base/max/min LR:", BASE_LR, "/", MAX_LR, "/", MIN_LR)
print("  weight decay   :", WEIGHT_DECAY)
print("  grad clip      :", GRAD_CLIP)


# ============================================================
# 11. METRICS
# ============================================================
def summarize_array(arr, scale=1.0):
    arr = np.asarray(arr, dtype=np.float64) * scale
    return {
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "std": float(np.std(arr)),
    }


def short_summary_str(d):
    return (
        f"min={d['min']:.3f}, max={d['max']:.3f}, "
        f"mean={d['mean']:.3f}, median={d['median']:.3f}, std={d['std']:.3f}"
    )


@torch.no_grad()
def evaluate_monitor(cases):
    model.eval()
    mse_all, rel_all, r2_all = [], [], []

    for start in range(0, len(cases), EVAL_BATCH_SIZE):
        sub = cases[start:start + EVAL_BATCH_SIZE]
        batch = make_eval_batch(sub)

        pred_all_n = model(batch["mat_ctx"], batch["row_types"])
        pred_n = pred_all_n[:, -1, :]
        pred = pred_n * uf_std + uf_mean
        target = batch["target"]

        mse_i = torch.mean((pred - target) ** 2, dim=1)
        rel_i = (
            torch.linalg.norm(pred - target, dim=1)
            / torch.linalg.norm(target, dim=1).clamp_min(EPS)
        )

        y_mean = torch.mean(target, dim=1, keepdim=True)
        sse = torch.sum((pred - target) ** 2, dim=1)
        sst = torch.sum((target - y_mean) ** 2, dim=1).clamp_min(EPS)
        r2_i = 1.0 - sse / sst

        mse_all.append(mse_i.cpu().numpy())
        rel_all.append(rel_i.cpu().numpy())
        r2_all.append(r2_i.cpu().numpy())

    return {
        "MSE_mean": float(np.mean(np.concatenate(mse_all))),
        "RelL2_summary_%": summarize_array(np.concatenate(rel_all), 100.0),
        "R2_summary": summarize_array(np.concatenate(r2_all), 1.0),
    }


# ============================================================
# 12. OUTPUT FILES
# ============================================================
FINAL_CKPT = SAVE_DIR / f"{RUN_TAG}_final_epoch300.pt"
BEST_MONITOR_CKPT = SAVE_DIR / f"{RUN_TAG}_best_trainmonitor.pt"
HISTORY_JSON = SAVE_DIR / f"{RUN_TAG}_history.json"
HISTORY_NPZ = SAVE_DIR / f"{RUN_TAG}_history_arrays.npz"
SPLIT_NPZ = SAVE_DIR / f"{RUN_TAG}_split.npz"

np.savez(SPLIT_NPZ, train_ops=train_ops, test_ops=test_ops)


# ============================================================
# 13. EPOCH-0 MONITOR
# ============================================================
history = []
initial_monitor = evaluate_monitor(monitor_cases)
history.append({
    "epoch": 0,
    "trainLoss_norm": None,
    "monitorMSE": initial_monitor["MSE_mean"],
    "monitorRelL2_summary_%": initial_monitor["RelL2_summary_%"],
    "monitorR2_summary": initial_monitor["R2_summary"],
    "seconds": 0.0,
    "lr": float(opt.param_groups[0]["lr"]),
})

print("\n=== EPOCH 0 TRAIN-SIDE K=5 MONITOR ===")
print("Monitor RelL2%:", short_summary_str(initial_monitor["RelL2_summary_%"]))
print("Monitor R2    :", short_summary_str(initial_monitor["R2_summary"]))


# ============================================================
# 14. TRAIN FIXED 300 EPOCHS
# ============================================================
global_step = 0
k_episode_counts = {k: 0 for k in range(MIN_DEMOS, MAX_DEMOS + 1)}
best_monitor_mean = float("inf")
best_monitor_epoch = None

for ep in range(1, EPOCHS + 1):
    model.train()
    loss_list = []
    epoch_k_counts = {k: 0 for k in range(MIN_DEMOS, MAX_DEMOS + 1)}
    t0 = time.time()

    for _ in range(STEPS_PER_EPOCH):
        global_step += 1
        batch = make_train_batch(BATCH_SIZE)

        for K, n_k in batch["K_counts"].items():
            k_episode_counts[K] += n_k
            epoch_k_counts[K] += n_k

        opt.zero_grad(set_to_none=True)

        # ONE forward pass for the whole heterogeneous logical batch.
        pred_all_n = model(
            batch["mat_ctx"],
            batch["row_types"],
            src_key_padding_mask=batch["padding_mask"],
        )

        # Query position depends on K: j_i = (3*K_i + 2) - 1.
        batch_ids = torch.arange(BATCH_SIZE, device=DEVICE)
        pred_n = pred_all_n[
            batch_ids,
            batch["lengths"] - 1,
            :,
        ]
        loss = mse(pred_n, batch["target_n"])

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
        scheduler.step()
        loss_list.append(float(loss.item()))

    train_loss_norm = float(np.mean(loss_list))
    should_monitor = (ep == 1) or (ep % PRINT_EVERY == 0) or (ep == EPOCHS)

    if should_monitor:
        monitor = evaluate_monitor(monitor_cases)
        monitor_mean = float(monitor["RelL2_summary_%"]["mean"])
        elapsed = float(time.time() - t0)

        row = {
            "epoch": ep,
            "trainLoss_norm": train_loss_norm,
            "monitorMSE": monitor["MSE_mean"],
            "monitorRelL2_summary_%": monitor["RelL2_summary_%"],
            "monitorR2_summary": monitor["R2_summary"],
            "seconds": elapsed,
            "lr": float(opt.param_groups[0]["lr"]),
            "epoch_K_episode_counts": {
                str(k): int(v) for k, v in epoch_k_counts.items()
            },
        }
        history.append(row)

        print(
            f"\nepoch {ep:03d} | trainLoss(norm) {train_loss_norm:.6f} | "
            f"monitor RelL2 {monitor_mean:.4f}% | {elapsed:.1f}s | "
            f"lr {row['lr']:.6e}"
        )
        print("  Monitor RelL2%:", short_summary_str(monitor["RelL2_summary_%"]))
        print("  Monitor R2    :", short_summary_str(monitor["R2_summary"]))
        print("  K episode counts this epoch:", epoch_k_counts)

        if monitor_mean < best_monitor_mean:
            best_monitor_mean = monitor_mean
            best_monitor_epoch = ep

            torch.save({
                "epoch": ep,
                "best_monitor_relL2_mean_percent": best_monitor_mean,
                "model": model.state_dict(),
                "optimizer": opt.state_dict(),
                "scheduler": scheduler.state_dict(),
                "norm": {
                    "x_mean": x_mean,
                    "x_std": x_std,
                    "uc_mean": uc_mean,
                    "uc_std": uc_std,
                    "uf_mean": uf_mean,
                    "uf_std": uf_std,
                },
                "config": {
                    "RUN_TAG": RUN_TAG,
                    "MAT_PATH": str(MAT_PATH),
                    "K_BATCHING": "mixed_K_pad_to_17_one_forward",
                    "MIN_DEMOS": MIN_DEMOS,
                    "MAX_DEMOS": MAX_DEMOS,
                    "TEST_FIXED_DEMOS": TEST_FIXED_DEMOS,
                    "NX": NX,
                    "N_TRAIN_VAR_PER_FAM": N_TRAIN_VAR_PER_FAM,
                    "N_TEST_VAR_PER_FAM": N_TEST_VAR_PER_FAM,
                    "train_ops": train_ops,
                    "test_ops": test_ops,
                    "BATCH_SIZE": BATCH_SIZE,
                    "EPOCHS": EPOCHS,
                    "STEPS_PER_EPOCH": STEPS_PER_EPOCH,
                    "WEIGHT_DECAY": WEIGHT_DECAY,
                    "BASE_LR": BASE_LR,
                    "MAX_LR": MAX_LR,
                    "MIN_LR": MIN_LR,
                    "WARMUP_FRAC": WARMUP_FRAC,
                    "GRAD_CLIP": GRAD_CLIP,
                    "D_MODEL": D_MODEL,
                    "N_HEAD": N_HEAD,
                    "N_LAYERS": N_LAYERS,
                    "D_FF": D_FF,
                    "DROPOUT": DROPOUT,
                    "N_ROW_TYPES": N_ROW_TYPES,
                    "MAX_CTX_ROWS": MAX_CTX_ROWS,
                    "SEED": SEED,
                    "MIXED_K": True,
                    "FORWARD_BACKWARD_PER_OPTIMIZER_STEP": 1,
                    "PADDING": "right-pad all sequences to 17 rows",
                    "PADDING_MASK": (
                        "torch src_key_padding_mask; True means padded key/value"
                    ),
                    "DATASET_PREPROCESSING": (
                        "coarse states already stored after direct linear "
                        "interpolation 50->100; no interpolation in training code"
                    ),
                },
            }, BEST_MONITOR_CKPT)

            print(
                "  NEW BEST TRAIN MONITOR:",
                f"{best_monitor_mean:.6f}% at epoch {best_monitor_epoch}"
            )

        with open(HISTORY_JSON, "w") as f:
            json.dump(history, f, indent=2)


# ============================================================
# 15. SAVE FINAL EPOCH-300 CHECKPOINT
# ============================================================
final_checkpoint = {
    "epoch": EPOCHS,
    "best_monitor_epoch": best_monitor_epoch,
    "best_monitor_relL2_mean_percent": best_monitor_mean,
    "model": model.state_dict(),
    "optimizer": opt.state_dict(),
    "scheduler": scheduler.state_dict(),
    "norm": {
        "x_mean": x_mean,
        "x_std": x_std,
        "uc_mean": uc_mean,
        "uc_std": uc_std,
        "uf_mean": uf_mean,
        "uf_std": uf_std,
    },
    "config": {
        "RUN_TAG": RUN_TAG,
        "MAT_PATH": str(MAT_PATH),
        "K_BATCHING": "mixed_K_pad_to_17_one_forward",
        "MIN_DEMOS": MIN_DEMOS,
        "MAX_DEMOS": MAX_DEMOS,
        "TEST_FIXED_DEMOS": TEST_FIXED_DEMOS,
        "NX": NX,
        "N_TRAIN_VAR_PER_FAM": N_TRAIN_VAR_PER_FAM,
        "N_TEST_VAR_PER_FAM": N_TEST_VAR_PER_FAM,
        "train_ops": train_ops,
        "test_ops": test_ops,
        "BATCH_SIZE": BATCH_SIZE,
        "EPOCHS": EPOCHS,
        "STEPS_PER_EPOCH": STEPS_PER_EPOCH,
        "WEIGHT_DECAY": WEIGHT_DECAY,
        "BASE_LR": BASE_LR,
        "MAX_LR": MAX_LR,
        "MIN_LR": MIN_LR,
        "WARMUP_FRAC": WARMUP_FRAC,
        "GRAD_CLIP": GRAD_CLIP,
        "D_MODEL": D_MODEL,
        "N_HEAD": N_HEAD,
        "N_LAYERS": N_LAYERS,
        "D_FF": D_FF,
        "DROPOUT": DROPOUT,
        "N_ROW_TYPES": N_ROW_TYPES,
        "MAX_CTX_ROWS": MAX_CTX_ROWS,
        "SEED": SEED,
        "MIXED_K": True,
        "FORWARD_BACKWARD_PER_OPTIMIZER_STEP": 1,
        "PADDING": "right-pad all sequences to 17 rows",
        "PADDING_MASK": "torch src_key_padding_mask; True means padded key/value",
        "DATASET_PREPROCESSING": (
            "coarse states already stored after direct linear interpolation "
            "50->100; no interpolation in training code"
        ),
    },
    "K_episode_counts": {str(k): int(v) for k, v in k_episode_counts.items()},
}

torch.save(final_checkpoint, FINAL_CKPT)

with open(HISTORY_JSON, "w") as f:
    json.dump(history, f, indent=2)

np.savez(
    HISTORY_NPZ,
    epochs=np.asarray([h["epoch"] for h in history], dtype=np.int64),
    monitor_relL2_mean_percent=np.asarray(
        [h["monitorRelL2_summary_%"]["mean"] for h in history], dtype=np.float64
    ),
    monitor_r2_mean=np.asarray(
        [h["monitorR2_summary"]["mean"] for h in history], dtype=np.float64
    ),
)

print("\n" + "=" * 110)
print("BURGERS MIXED-K PAD-TO-17 TRAINING COMPLETE")
print("=" * 110)
print("Final epoch                  :", EPOCHS)
print("Best train-monitor epoch     :", best_monitor_epoch)
print("Best train-monitor RelL2     :", f"{best_monitor_mean:.6f}%")
print("\nK episode counts:")
for k in range(MIN_DEMOS, MAX_DEMOS + 1):
    print(f"  K={k}: {k_episode_counts[k]}")
print("\nSaved:")
print(" -", FINAL_CKPT)
print(" -", BEST_MONITOR_CKPT)
print(" -", HISTORY_JSON)
print(" -", HISTORY_NPZ)
print(" -", SPLIT_NPZ)
print("\nNo held-out test/OOD evaluation was run during training.")
print("For the clean old-vs-new comparison, evaluate the FINAL epoch-300")
print("checkpoint on the exact same fixed 1024 held-out K=5 cases.")
print("\nDONE.")


BURGERS STANDARD DIRECT-LINEAR TRAINING -- MIXED K, PAD ALL TO 17
DEVICE   : cuda
MAT_PATH : /kaggle/input/notebooks/xinyuanli913/dataset-box-ood/all8families_box_linear_id_ampood_recorded.mat
SAVE_DIR : /kaggle/working/burgers_box_linear_mixedK_pad17
K rule   : each episode independently samples K in {1,...,5}
Batching  : pad every sequence to 17; ONE forward/backward per optimizer step

Loaded dataset:
  uc / uf / u0 : (8, 100, 100, 100)
  operators     : (8, 100, 3)
  x             : (100,)
  families      : 8
  operators/fam : 100
  ICs/operator  : 100
  model row Nr  : 100
  preprocessing : dataset already direct-linear 50 -> 100
                  NO interpolation in this training script

Operator split:
  train operators: 560
  test operators : 240
  per family     : 70 train / 30 held-out

Normalization from ID training operators only:
  x  mean/std : 0.49500000 / 0.28866172
  uc mean/std : 0.22864805 / 0.22771730
  uf mean/std : 0.22981256 / 0.22123329


/tmp/ipykernel_23/4219067757.py:475: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)



Model:
  row width       : 100
  d_model         : 96
  heads           : 4
  layers          : 3
  FF              : 192
  dropout         : 0.1
  trainable params: 255,364

Optimizer:
  AdamW
  batch size     : 1000
  epochs         : 300
  steps/epoch    : 1200
  total steps    : 360000
  warmup steps   : 28800
  base/max/min LR: 0.0001 / 0.0002 / 1e-05
  weight decay   : 0.0001
  grad clip      : 1.0

=== EPOCH 0 TRAIN-SIDE K=5 MONITOR ===
Monitor RelL2%: min=27.703, max=141.582, mean=68.386, median=65.349, std=22.084
Monitor R2    : min=-21.247, max=0.033, mean=-1.333, median=-0.454, std=2.446

epoch 001 | trainLoss(norm) 0.286343 | monitor RelL2 19.1034% | 48.8s | lr 1.041667e-04
  Monitor RelL2%: min=2.500, max=52.607, mean=19.103, median=18.563, std=9.583
  Monitor R2    : min=-0.015, max=0.987, mean=0.853, median=0.878, std=0.109
  K episode counts this epoch: {1: 239622, 2: 240399, 3: 240324, 4: 239781, 5: 239874}
  NEW BEST TRAIN MONITOR: 19.103366% at epoch 1

epoch 005 | 